# Experiment E6: In-Depth Evaluation & Analysis of Best Model
This notebook provides granular analysis of the winning configuration on the UAV enemy detection test split:
- Precision, Recall, mAP@0.5, mAP@0.5:0.95 overall and per-class
- Normalized & raw Confusion Matrices
- Inference speed and FPS latency benchmarking
- Sample detection visualization on challenging frames


In [ ]:
# 1. Environment & Setup
import sys
from pathlib import Path
import torch
import ultralytics
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.modules.cbam import register_cbam
register_cbam()

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


In [ ]:
# 2. Locate Best Model Checkpoint
from experiments.scripts.analyze_e6 import find_experiment_runs, parse_run_results

project_dir = PROJECT_ROOT / "runs" / "experiments"
runs = find_experiment_runs(project_dir)

if not runs:
    raise FileNotFoundError(f"No experiment runs found in {project_dir}. Train E0-E5 first.")

rows = []
for exp_id, r_dir in runs.items():
    data = parse_run_results(r_dir)
    data["Experiment"] = exp_id
    rows.append(data)

df_runs = pd.DataFrame(rows).sort_values("val_map50", ascending=False).reset_index(drop=True)
best_exp = df_runs.iloc[0]
BEST_WEIGHTS = Path(best_exp["weights_path"]).resolve()

print(f"Winning Experiment : {best_exp['Experiment']}")
print(f"Validation mAP@0.5  : {best_exp['val_map50']:.4f}")
print(f"Weights Checkpoint : {BEST_WEIGHTS}")
display(df_runs[["Experiment", "val_precision", "val_recall", "val_map50", "val_map50_95"]])


In [ ]:
# 3. Test Split Evaluation
from experiments.scripts.evaluate import evaluate_model

DATASET_YAML = PROJECT_ROOT / "datasets" / "enemy-detection-1-fps-v1" / "data_yolo26_local.yaml"
if not DATASET_YAML.is_file():
    DATASET_YAML = PROJECT_ROOT / "datasets" / "enemy-detection-1-fps-v1" / "data.yaml"

test_eval = evaluate_model(
    weights_path=BEST_WEIGHTS,
    data_yaml=DATASET_YAML,
    split="test",
    device=DEVICE,
    imgsz=640
)


In [ ]:
# 4. Latency & Throughput Benchmark
from experiments.scripts.analyze_e6 import benchmark_latency
from ultralytics import YOLO

model = YOLO(str(BEST_WEIGHTS))
bench = benchmark_latency(model, imgsz=640, device="cuda" if torch.cuda.is_available() else "cpu")
print("=== SPEED BENCHMARK ===")
print(f"Inference Latency: {bench['latency_ms']} ms/frame")
print(f"Throughput       : {bench['fps']} FPS")


In [ ]:
# 5. Visual Predictions on Sample Test Images
import random

SPLIT_DIR = PROJECT_ROOT / "datasets" / "enemy-detection-1-fps-v1" / "test" / "images"
if not SPLIT_DIR.is_dir():
    SPLIT_DIR = PROJECT_ROOT / "datasets" / "enemy-detection-1-fps-v1" / "valid" / "images"

if SPLIT_DIR.is_dir():
    all_images = sorted(list(SPLIT_DIR.glob("*.jpg")) + list(SPLIT_DIR.glob("*.png")))
    sample_images = random.sample(all_images, min(6, len(all_images)))

    results = model.predict(
        source=[str(p) for p in sample_images],
        conf=0.25,
        device=DEVICE,
        save=False
    )

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for ax, res in zip(axes, results):
        plotted = res.plot(pil=True, labels=True, boxes=True)
        ax.imshow(plotted)
        ax.set_title(Path(res.path).name, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
